In [1]:
import dgl.nn as dglnn
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import category_encoders as ce
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn import metrics

In [2]:
data = pd.read_csv('CSE-IDS/NF-CSE-CIC-IDS2018-v2.csv').sample(n=7000000)
#data = pd.read_csv('NF-CSE-CIC-IDS2018.csv')

In [3]:
#df = pd.read_csv('CSE-IDS/NF-CSE-CIC-IDS2018-v2.csv').sample(n=100000)

In [4]:
#data

In [5]:
#data, NAlist = reduce_mem_usage(data)
#print("_________________")
#print("")
#print("Warning: the following columns have missing values filled with 'df['column_name'].min() -1': ")
#print("_________________")
#print("")
#print(NAlist)



In [6]:
data

,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,...,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
11634926,172.31.64.72,59860,172.31.0.2,53,17,0.000,78,1,206,1,...,0,0,0,0,23240,1,59,0,0,Benign
18574274,180.179.21.221,58296,172.31.64.80,3389,6,0.000,1480,8,1913,8,...,8192,64000,0,0,0,0,0,0,0,Benign
7749707,172.31.66.100,61167,172.31.0.2,53,17,0.000,75,1,91,1,...,0,0,0,0,58349,1,37,0,0,Benign
436786,172.31.64.81,51972,13.32.233.249,443,6,91.178,1001,11,6261,10,...,8192,29200,0,0,0,0,0,0,0,Benign
7702096,172.31.67.52,63475,172.31.0.2,53,17,5.126,63,1,115,1,...,0,0,0,0,19053,1,60,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026686,172.31.64.23,58865,172.31.0.2,53,17,0.000,140,2,366,2,...,0,0,0,0,63206,1,60,0,0,Benign
13189591,18.218.229.235,57748,172.31.69.28,80,6,7.000,480,5,1147,5,...,65535,26883,0,0,0,0,0,0,1,DDOS attack-HOIC
6272678,172.31.68.9,51073,172.31.0.2,53,17,0.000,124,2,188,2,...,0,0,0,0,17545,1,60,0,0,Benign
3188198,172.31.65.6,63131,172.31.0.2,53,17,0.000,70,1,170,1,...,0,0,0,0,26217,1,60,0,0,Benign


In [7]:
data.Attack.value_counts()

Benign                      6163801
DDOS attack-HOIC             400608
DoS attacks-Hulk             160169
DDoS attacks-LOIC-HTTP       113834
Bot                           52745
Infilteration                 43050
SSH-Bruteforce                35027
DoS attacks-GoldenEye         10309
FTP-BruteForce                 9553
DoS attacks-SlowHTTPTest       5271
DoS attacks-Slowloris          3537
Brute Force -Web                813
DDOS attack-LOIC-UDP            745
Brute Force -XSS                369
SQL Injection                   169
Name: Attack, dtype: int64

In [8]:
data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))


In [9]:
data['IPV4_SRC_ADDR'] = data.IPV4_SRC_ADDR.apply(str)
data['L4_SRC_PORT'] = data.L4_SRC_PORT.apply(str)
data['IPV4_DST_ADDR'] = data.IPV4_DST_ADDR.apply(str)
data['L4_DST_PORT'] = data.L4_DST_PORT.apply(str)

In [10]:
data['IPV4_SRC_ADDR'] = data['IPV4_SRC_ADDR'] + ':' + data['L4_SRC_PORT']
data['IPV4_DST_ADDR'] = data['IPV4_DST_ADDR'] + ':' + data['L4_DST_PORT']

In [11]:
data.drop(columns=['L4_SRC_PORT','L4_DST_PORT'],inplace=True)

In [12]:


data



,IPV4_SRC_ADDR,IPV4_DST_ADDR,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
11634926,172.18.76.78:59860,172.31.0.2:53,17,0.000,78,1,206,1,0,0,...,0,0,0,0,23240,1,59,0,0,Benign
18574274,172.19.125.124:58296,172.31.64.80:3389,6,0.000,1480,8,1913,8,222,222,...,8192,64000,0,0,0,0,0,0,0,Benign
7749707,172.19.162.231:61167,172.31.0.2:53,17,0.000,75,1,91,1,0,0,...,0,0,0,0,58349,1,37,0,0,Benign
436786,172.16.65.44:51972,13.32.233.249:443,6,91.178,1001,11,6261,10,223,223,...,8192,29200,0,0,0,0,0,0,0,Benign
7702096,172.20.61.3:63475,172.31.0.2:53,17,5.126,63,1,115,1,0,0,...,0,0,0,0,19053,1,60,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026686,172.23.252.131:58865,172.31.0.2:53,17,0.000,140,2,366,2,0,0,...,0,0,0,0,63206,1,60,0,0,Benign
13189591,172.29.43.122:57748,172.31.69.28:80,6,7.000,480,5,1147,5,219,219,...,65535,26883,0,0,0,0,0,0,1,DDOS attack-HOIC
6272678,172.17.184.205:51073,172.31.0.2:53,17,0.000,124,2,188,2,0,0,...,0,0,0,0,17545,1,60,0,0,Benign
3188198,172.19.187.85:63131,172.31.0.2:53,17,0.000,70,1,170,1,0,0,...,0,0,0,0,26217,1,60,0,0,Benign


In [13]:
data.drop(columns=['Attack'],inplace = True)


In [14]:
data.rename(columns={"Label": "label"},inplace = True)

In [15]:
label = data.label

In [16]:
data.drop(columns=['label'],inplace = True)

In [17]:
scaler = StandardScaler()

data =  pd.concat([data, label], axis=1)

data

,IPV4_SRC_ADDR,IPV4_DST_ADDR,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,label
11634926,172.18.76.78:59860,172.31.0.2:53,17,0.000,78,1,206,1,0,0,...,0,0,0,0,0,23240,1,59,0,0
18574274,172.19.125.124:58296,172.31.64.80:3389,6,0.000,1480,8,1913,8,222,222,...,1,8192,64000,0,0,0,0,0,0,0
7749707,172.19.162.231:61167,172.31.0.2:53,17,0.000,75,1,91,1,0,0,...,0,0,0,0,0,58349,1,37,0,0
436786,172.16.65.44:51972,13.32.233.249:443,6,91.178,1001,11,6261,10,223,223,...,4,8192,29200,0,0,0,0,0,0,0
7702096,172.20.61.3:63475,172.31.0.2:53,17,5.126,63,1,115,1,0,0,...,0,0,0,0,0,19053,1,60,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026686,172.23.252.131:58865,172.31.0.2:53,17,0.000,140,2,366,2,0,0,...,0,0,0,0,0,63206,1,60,0,0
13189591,172.29.43.122:57748,172.31.69.28:80,6,7.000,480,5,1147,5,219,219,...,0,65535,26883,0,0,0,0,0,0,1
6272678,172.17.184.205:51073,172.31.0.2:53,17,0.000,124,2,188,2,0,0,...,0,0,0,0,0,17545,1,60,0,0
3188198,172.19.187.85:63131,172.31.0.2:53,17,0.000,70,1,170,1,0,0,...,0,0,0,0,0,26217,1,60,0,0


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
     data, label, test_size=0.3, random_state=123,stratify= label)

In [19]:
encoder = ce.TargetEncoder(cols=['TCP_FLAGS','L7_PROTO','PROTOCOL',
                                  'CLIENT_TCP_FLAGS','SERVER_TCP_FLAGS','ICMP_TYPE',
                                  'ICMP_IPV4_TYPE','DNS_QUERY_ID','DNS_QUERY_TYPE',
                                  'FTP_COMMAND_RET_CODE'])

In [20]:
encoder.fit(X_train, y_train)
X_train = encoder.transform(X_train)

In [21]:
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)

In [22]:
X_train.dropna(inplace=True)

In [23]:
cols_to_norm = list(set(list(X_train.iloc[:, 2:].columns ))  - set(list(['label'])) )
X_train[cols_to_norm] = scaler.fit_transform(X_train[cols_to_norm])

/home/tselab/anaconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:1066: RuntimeWarning: overflow encountered in square
  temp **= 2
/home/tselab/anaconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:1072: RuntimeWarning: overflow encountered in square
  new_unnormalized_variance -= correction**2 / new_sample_count
/home/tselab/anaconda3/lib/python3.10/site-packages/sklearn/utils/extmath.py:1072: RuntimeWarning: invalid value encountered in subtract
  new_unnormalized_variance -= correction**2 / new_sample_count
/home/tselab/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_data.py:86: RuntimeWarning: overflow encountered in square
  upper_bound = n_samples * eps * var + (n_samples * mean * eps) ** 2


In [24]:
X_train['h'] = X_train[ cols_to_norm ].values.tolist()

In [25]:
X_train

,IPV4_SRC_ADDR,IPV4_DST_ADDR,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,...,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,label,h
446229,172.26.45.67:61693,172.31.0.2:53,-1.186847,-0.434101,-0.025317,-0.020381,-0.013666,-0.026436,-0.574877,-0.718415,...,-0.602456,-0.837209,-0.020446,-0.020476,-1.253050,-1.203599,1.396415,1.221245e-15,0,"[-0.7663797657657416, -0.018549186422365485, -..."
14290599,172.20.217.134:25247,172.31.66.12:23,0.842680,-0.467333,-0.025138,-0.019436,-0.013977,-0.029514,-0.620401,-0.345917,...,1.281445,-0.837209,-0.020446,-0.020476,0.823088,0.830814,-0.726253,1.221245e-15,0,"[-0.9673960156356111, -0.018549186422365485, -..."
4850927,172.22.62.190:56455,172.31.0.2:53,-1.186847,-0.434101,-0.025258,-0.020381,-0.013709,-0.026436,-0.574877,-0.718415,...,-0.602456,-0.837209,-0.020446,-0.020476,-1.406078,-1.203599,-0.006062,1.221245e-15,0,"[-0.8051076671168175, -0.018549186422365485, -..."
11111924,172.25.81.229:59223,172.31.66.80:3389,0.842680,-0.434101,-0.004288,-0.013762,-0.010069,-0.007964,-0.592212,-0.610297,...,-0.173180,1.718689,-0.020446,-0.020476,0.823088,0.830814,-0.726253,1.221245e-15,0,"[1.1958339026887632, -0.018549186422365485, -0..."
2479570,172.29.17.25:53328,209.85.202.94:443,0.842680,-0.458714,-0.012569,-0.011871,-0.003791,-0.001806,0.189683,-0.769225,...,-0.173180,0.871249,-0.020446,-0.020476,0.823088,0.830814,-0.726253,1.221245e-15,0,"[1.6697896477947856, -0.018549186422365485, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426280,172.17.10.167:53610,196.25.125.65:443,0.842680,-0.434101,-0.004333,-0.010925,-0.001886,0.010509,2.114406,1.650228,...,-0.173180,-0.254144,-1.479495,-1.480678,0.823088,0.830814,-0.726253,1.221245e-15,0,"[1.725115221153465, -0.018549186422365485, -1...."
3292508,172.26.209.126:64119,172.31.64.125:3389,0.842680,-0.434101,-0.003692,-0.012816,-0.010069,-0.007964,-0.592212,-0.610297,...,-0.173180,1.718689,-0.020446,-0.020476,0.823088,0.830814,-0.726253,1.221245e-15,0,"[1.1958339026887632, -0.018549186422365485, -0..."
15475490,172.29.157.215:58435,172.31.0.2:53,-1.186847,-0.431365,-0.025302,-0.020381,-0.013774,-0.026436,-0.574877,-0.718415,...,-0.602456,-0.837209,-0.020446,-0.020476,-1.190778,-1.203599,1.548034,1.221245e-15,0,"[-0.8622774262541197, -0.018549186422365485, -..."
6985079,172.19.122.15:65139,172.31.0.2:53,-1.186847,-0.434101,-0.025451,-0.020381,-0.013787,-0.026436,-0.574877,-0.718415,...,-0.602456,-0.837209,-0.020446,-0.020476,-0.731770,-1.202872,-0.536729,1.221245e-15,0,"[-0.8733425409258556, -0.018549186422365485, -..."


In [26]:
G = nx.from_pandas_edgelist(X_train, "IPV4_SRC_ADDR", "IPV4_DST_ADDR", ['h','label'],create_using=nx.MultiGraph())


In [ ]:
G = G.to_directed()

G = from_networkx(G,edge_attrs=['h','label'] )



In [ ]:
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1])


In [ ]:
G.ndata['h'] = th.reshape(G.ndata['h'], (G.ndata['h'].shape[0], 1,G.ndata['h'].shape[1]))
G.edata['h'] = th.reshape(G.edata['h'], (G.edata['h'].shape[0], 1,G.edata['h'].shape[1]))


In [ ]:
G.edata['train_mask'] = th.ones(len(G.edata['h']), dtype=th.bool)


In [ ]:
class Model(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super().__init__()
        self.gnn = SAGE(ndim_in, ndim_out, edim, activation, dropout)
        self.pred = MLPPredictor(ndim_out, 2)
    def forward(self, g, nfeats, efeats):
        h = self.gnn(g, nfeats, efeats)
        return self.pred(g, h)



In [ ]:
class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        self.activation = activation

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.src['h'], edges.data['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
            return g.ndata['h']


class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super(SAGE, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(SAGELayer(ndim_in, edim, 128, activation))
        self.layers.append(SAGELayer(128, edim, ndim_out, activation))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, g, nfeats, efeats):
        for i, layer in enumerate(self.layers):
            if i != 0:
                nfeats = self.dropout(nfeats)
            nfeats = layer(g, nfeats, efeats)
        return nfeats.sum(1)


In [ ]:
class MLPPredictor(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        global score
        global emb
        emb = th.cat([h_u, h_v], 1)
        score = self.W(th.cat([h_u, h_v], 1))
        return {'score': score}

    def forward(self, graph, h):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']


In [ ]:
G.device

In [ ]:
G.ndata['h'].device

In [ ]:
G.edata['h'].device

In [ ]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(G.edata['label'].cpu().numpy()),
                                                 y=G.edata['label'].cpu().numpy())

In [ ]:
class_weights = th.FloatTensor(class_weights)#.cuda()
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [ ]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

In [ ]:
import os
node_features = G.ndata['h']
edge_features = G.edata['h']

edge_label = G.edata['label']
train_mask = G.edata['train_mask']

model = Model(G.ndata['h'].shape[2], 128, G.ndata['h'].shape[2], F.relu, 0.2)#.cuda()
opt = th.optim.Adam(model.parameters())

In [ ]:
import os
os.remove("cluster_gcn.pkl")

In [ ]:
import dgl
import dgl.nn as dglnn
import torch

In [ ]:
num_parts = 1000
sampler = dgl.dataloading.ClusterGCNSampler(G, num_parts, prefetch_edata=["h", "label", "train_mask"]#, "val_mask", "test_mask"]
)

In [ ]:
dataloader = dgl.dataloading.DataLoader(
    G, torch.arange(num_parts), sampler,
    batch_size=20, shuffle=True, drop_last=False, num_workers=4)

In [ ]:
import time
for _ in range(10):
    t0 = time.time()
    history = model.train()
    for it, sg in enumerate(dataloader):
        x = sg.edata["h"]
        x2 = sg.ndata["h"]
        y = sg.edata["label"]
        m = sg.edata["train_mask"].bool()
        y_hat = model(sg, x2, x)
        loss = F.cross_entropy(y_hat[m], y[m])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if it % 20 == 0:
            #acc = MF.accuracy(y_hat[m], y[m])
            #mem = torch.cuda.max_memory_allocated() / 1000000
            #print("Loss", loss.item(), "Acc", acc.item(), "GPU Mem", mem, "MB")
            print('Epoch:', it ,' Training acc:', compute_accuracy(y_hat[m], y[m]))
           # history = model.fit(y_hat[m], y[m], epochs=5, batch_size=1)
    tt = time.time()
    print(tt - t0)
    #durations.append(tt - t0)

In [ ]:
X_test = encoder.transform(X_test)

X_test[cols_to_norm] = scaler.transform(X_test[cols_to_norm])



In [ ]:
X_test['h'] = X_test[ cols_to_norm ].values.tolist()


In [ ]:
G_test = nx.from_pandas_edgelist(X_test, "IPV4_SRC_ADDR", "IPV4_DST_ADDR", ['h','label'],create_using=nx.MultiGraph())
G_test = G_test.to_directed()
G_test = from_networkx(G_test,edge_attrs=['h','label'] )
#actual = G_test.edata.pop('label')
G_test.ndata['feature'] = th.ones(G_test.num_nodes(),G.ndata['h'].shape[2])

In [ ]:
G_test.ndata['feature'] = th.reshape(G_test.ndata['feature'], (G_test.ndata['feature'].shape[0], 1, G_test.ndata['feature'].shape[1]))
G_test.edata['h'] = th.reshape(G_test.edata['h'], (G_test.edata['h'].shape[0], 1, G_test.edata['h'].shape[1]))



In [ ]:
G_test.edata['test_mask'] = th.ones(len(G_test.edata['h']), dtype=th.bool)

In [ ]:
import os
os.remove("cluster_gcn.pkl")

In [ ]:
num_parts = 2000
sampler = dgl.dataloading.ClusterGCNSampler(G_test, num_parts, prefetch_edata=["h", "label", "test_mask"]#, "val_mask", "test_mask"]
)

In [ ]:
dataloader = dgl.dataloading.DataLoader(
    G_test, torch.arange(num_parts), sampler,
    batch_size=20, shuffle=True, drop_last=False, num_workers=4)

In [ ]:
model.eval()

In [ ]:
import torchmetrics.functional as MF

In [ ]:
import timeit
totaltime=0
start_time = timeit.default_timer()
with torch.no_grad():
        val_preds, test_preds = [], []
        val_labels, test_labels = [], []
        for it, sg in enumerate(dataloader):
            x = sg.edata["h"]
            x2 = sg.ndata["feature"]
            y = sg.edata["label"]
            #m_val = sg.ndata["val_mask"].bool()
            m_test = sg.edata["test_mask"].bool()
            start_time = timeit.default_timer()
            y_hat = model(sg, x2, x)
            elapsed = timeit.default_timer() - start_time
            totaltime=totaltime+elapsed
           #val_preds.append(y_hat[m_val])
            #val_labels.append(y[m_val])
            test_preds.append(y_hat[m_test])
            test_labels.append(y[m_test])
            #break
       # val_preds = torch.cat(val_preds, 0)
       # val_labels = torch.cat(val_labels, 0)
        test_preds = torch.cat(test_preds, 0)
        test_labels = torch.cat(test_labels, 0)

        test_acc = MF.accuracy(
            test_preds,
            test_labels,
            task="multiclass",
           num_classes=2,
        )
        print("Test acc:", test_acc.item())
        #print("Test acc:", compute_accuracy(y_hat[m_test], y[m_test]))
        #tt = time.time()
        #print(tt - t0)ton_iot

In [ ]:
print(str(elapsed) + ' seconds')

In [ ]:
print(str(totaltime) + ' seconds')

In [ ]:
emb_viz=test_preds

In [ ]:
df_umap = pd.DataFrame(emb_viz, columns=['comp1', 'comp2'])

In [ ]:
test_preds = test_preds.argmax(1)
test_preds = th.Tensor.cpu(test_preds).detach().numpy()

In [ ]:
from sklearn.metrics import classification_report
target_names = ['Normal', 'Abnormal']
print(classification_report(test_labels, test_preds, digits=4,target_names=target_names))

In [ ]:
clf_report=classification_report(test_labels, test_preds, digits=4, output_dict=True, target_names=target_names)

In [ ]:
sns.heatmap(pd.DataFrame(clf_report).iloc[:-1, :].T, annot=True)